# Task 3: Dynamic Analysis
In this notebook, we track the structural evolution of each subreddit's interaction graph over time (weekly). We compute dynamic graph metrics such as density, modularity, and top leaders.


In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import os
import community.community_louvain as community_louvain 
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
print("Loading data...")
edges_df = pd.read_csv('graphs/all_edges_consolidated.csv')

# Ensure chronological order by sorting
edges_df.sort_values(by=['Subreddit', 'ISOYearWeek'], inplace=True)
subreddits = edges_df['Subreddit'].unique()

print(f"Found {len(subreddits)} subreddits over {edges_df['ISOYearWeek'].nunique()} unique weeks.")
edges_df.head()

In [ ]:
def compute_weekly_metrics(df_sub):
    weeks = sorted(df_sub['ISOYearWeek'].unique())
    weekly_metrics = []
    
    for week in weeks:
        df_week = df_sub[df_sub['ISOYearWeek'] == week]
        
        # Build Directed and Undirected Graph
        if len(df_week) < 2:
            continue
            
        G_directed = nx.DiGraph()
        for _, row in df_week.iterrows():
            G_directed.add_edge(row['Source'], row['Target'], weight=row['Weight'])
        G_directed.remove_edges_from(nx.selfloop_edges(G_directed))
        G_undirected = G_directed.to_undirected()
        
        # 1. Density
        density = nx.density(G_directed)
        
        # 2. Modularity
        try:
            partition = community_louvain.best_partition(G_undirected, weight='weight')
            modularity = community_louvain.modularity(partition, G_undirected)
        except:
            modularity = 0.0
            
        # 3. Top Leader (by PageRank)
        try:
            pagerank = nx.pagerank(G_directed, weight='weight')
            if len(pagerank) > 0:
                top_leader = max(pagerank, key=pagerank.get)
                top_score = pagerank[top_leader]
            else:
                top_leader = None
                top_score = 0.0
        except:
            top_leader = None
            top_score = 0.0
            
        weekly_metrics.append({
            'ISOYearWeek': week,
            'Num_Nodes': len(G_directed.nodes()),
            'Num_Edges': len(G_directed.edges()),
            'Density': density,
            'Modularity': modularity,
            'Top_Leader': top_leader,
            'Top_Leader_Score': top_score
        })
        
    return pd.DataFrame(weekly_metrics)

In [ ]:
all_dynamic_metrics = []

for subreddit in subreddits:
    print(f"Processing Subreddit: {subreddit}...")
    df_sub = edges_df[edges_df['Subreddit'] == subreddit]
    
    # Compute metrics for each available week
    weekly_metrics_df = compute_weekly_metrics(df_sub)
    if len(weekly_metrics_df) == 0:
        continue
        
    weekly_metrics_df['Subreddit'] = subreddit
    
    # Compute Week-over-Week Changes (Evolution Metrics)
    weekly_metrics_df['Delta_Density'] = weekly_metrics_df['Density'].diff().fillna(0)
    weekly_metrics_df['Delta_Modularity'] = weekly_metrics_df['Modularity'].diff().fillna(0)
    
    # Leader change flag
    weekly_metrics_df['Leader_Changed'] = (weekly_metrics_df['Top_Leader'] != weekly_metrics_df['Top_Leader'].shift(1)).astype(int)
    # The first week should technically not be a 'change' since there's no prior week, or we can leave it as 1 to indicate initiation.
    
    all_dynamic_metrics.append(weekly_metrics_df)
    
final_dynamic_df = pd.concat(all_dynamic_metrics, ignore_index=True)

# Rearrange columns
cols = ['Subreddit', 'ISOYearWeek', 'Num_Nodes', 'Num_Edges', 'Density', 'Delta_Density', 
        'Modularity', 'Delta_Modularity', 'Top_Leader', 'Top_Leader_Score', 'Leader_Changed']
final_dynamic_df = final_dynamic_df[cols]

# Save to CSV
final_dynamic_df.to_csv('graphs/dynamic_metrics.csv', index=False)
print("Finished processing temporal graphs and saved to 'graphs/dynamic_metrics.csv'.")
final_dynamic_df.head()

In [ ]:
# Let's plot the evolution of Modularity and Density for the largest subreddit
largest_sub = final_dynamic_df.groupby('Subreddit')['Num_Nodes'].sum().idxmax()
sub_data = final_dynamic_df[final_dynamic_df['Subreddit'] == largest_sub]

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.lineplot(data=sub_data, x='ISOYearWeek', y='Modularity', marker='o')
plt.xticks(rotation=45)
plt.title(f'Modularity Evolution in r/{largest_sub}')
plt.grid(True)

plt.subplot(1, 2, 2)
sns.lineplot(data=sub_data, x='ISOYearWeek', y='Density', marker='o', color='orange')
plt.xticks(rotation=45)
plt.title(f'Density Evolution in r/{largest_sub}')
plt.grid(True)

plt.tight_layout()
plt.show()